# SEA Pending Attrition/Transfer — Auto Reply

In [ ]:

SENDER_FILTER  = 'SEA_WFM_ID_Deletion@concentrix.com'
SUBJECT_FILTER = 'SEA - Client ID Deletion Notification'
ATTACH_KEYWORD = 'SEA - Pending Attrition'
PROCESS_FILTER = 'Expedia'
COUNTRY_FILTER = 'Vietnam'
TARGET_FOLDER  = 'GC3 + ExpAdmin'
DAYS_LOOKBACK  = 7

EMAIL_TO = 'SEA_WFM_ID_Deletion@concentrix.com;'
EMAIL_CC = (
    'Urmila Chakka <urmila.chakka1@concentrix.com>;'
    'Van Tran <van.tran@concentrix.com>;'
    'VN_HCM_ONE_EXP_WFM <VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com>;'
    'Puneet Suneja <puneet.suneja@concentrix.com>;'
    'KIRPAN PATAR <kirpan.patar@concentrix.com>;'
)

DISPLAY_PREVIEW = True

In [4]:
import os, re, time, json, pythoncom, win32com.client
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta, timezone
from IPython.display import display, HTML

HOME     = os.path.expanduser('~').replace('\\', '/')
SAVE_DIR = Path(HOME) / 'Downloads'
SAVE_DIR.mkdir(exist_ok=True)

# ── Helper: find column by keyword ───────────────────────────
def find_col(df, *keywords):
    for kw in keywords:
        m = next((c for c in df.columns if kw.lower() in c.lower()), None)
        if m: return m
    return None

# ── Helper: find Outlook subfolder ───────────────────────────
def find_named_folder(root_folders, name):
    for f in root_folders:
        try:
            if name.lower() in f.Name.lower(): return f
            for sub in f.Folders:
                if name.lower() in sub.Name.lower(): return sub
        except: pass
    return None

# ── Helper: fast email search with DASL ──────────────────────
def search_folder(folder, sender_kw, subj_kw, days=7, recurse=True):
    kw_smtp   = sender_kw.lower()
    kw_name   = sender_kw.split('@')[0].lower().replace('_', ' ')
    kw_nodash = sender_kw.split('@')[0].lower().replace('_', '')
    cutoff    = (datetime.now(timezone.utc) - timedelta(days=days)).strftime('%Y-%m-%d %H:%M')

    def sender_ok(item):
        sa = str(getattr(item, 'SenderEmailAddress', '') or '').lower()
        sn = str(getattr(item, 'SenderName',         '') or '').lower()
        return kw_smtp in sa or kw_name in sn or kw_nodash in sn.replace(' ', '')

    try:
        dasl = (
            f"@SQL=\"urn:schemas:httpmail:subject\" LIKE '%{subj_kw}%'"
            f" AND \"urn:schemas:httpmail:datereceived\" >= '{cutoff}'"
        )
        res = folder.Items.Restrict(dasl)
        res.Sort('[ReceivedTime]', True)
        for item in res:
            try:
                if sender_ok(item): return item
            except: continue
    except Exception as e:
        print(f'  [Restrict error] {e} — trying iteration')
        try:
            items = folder.Items
            items.Sort('[ReceivedTime]', True)
            for i, item in enumerate(items):
                if i >= 300: break
                try:
                    subj = str(getattr(item, 'Subject', '') or '')
                    if subj_kw.lower() in subj.lower() and sender_ok(item): return item
                except: continue
        except: pass

    if recurse:
        try:
            for sub in folder.Folders:
                found = search_folder(sub, sender_kw, subj_kw, days, recurse=False)
                if found: return found
        except: pass
    return None

# ── Helper: check if already replied ─────────────────────────
def already_replied(email_item, sent_folder):
    try:
        conv_id   = email_item.ConversationID
        orig_recv = email_item.ReceivedTime
        orig_subj = email_item.Subject
        subj_part = orig_subj[-20:] if len(orig_subj) > 20 else orig_subj
        dasl      = f"@SQL=\"urn:schemas:httpmail:subject\" LIKE '%{subj_part}%'"
        res       = sent_folder.Items.Restrict(dasl)
        res.Sort('[SentOn]', True)
        for sent in res:
            try:
                s_subj = str(getattr(sent, 'Subject', '') or '')
                s_conv = getattr(sent, 'ConversationID', None)
                s_on   = getattr(sent, 'SentOn', None)
                if not (s_conv == conv_id or
                        (s_subj.lower().startswith('re:') and subj_part.lower() in s_subj.lower())):
                    continue
                if s_on:
                    try:
                        r = orig_recv.replace(tzinfo=None) if getattr(orig_recv, 'tzinfo', None) else orig_recv
                        s = s_on.replace(tzinfo=None)      if getattr(s_on,      'tzinfo', None) else s_on
                        if s < r: continue
                    except: pass
                print(f'  [Reply check] Found: {s_subj!r} at {s_on}')
                return True, s_subj, s_on
            except: continue
    except Exception as e:
        print(f'[Reply check] Error: {e}')
    return False, None, None

# ── Helper: build pivot HTML table ───────────────────────────
def build_pivot_html(df, label, group_keywords):
    if df is None or df.empty: return ''
    gcols = [c for c in df.columns
             if any(k.lower() in c.lower() for k in group_keywords)]
    if not gcols:
        return f'<p style="color:#c00;font-family:Calibri,sans-serif;">No group cols for {label}</p>'

    df_grp = df[gcols].copy().fillna('').astype(str)
    pivot  = df_grp.groupby(gcols).size().reset_index(name='Count')
    total  = int(pivot['Count'].sum())

    P  = 'margin:0;padding:0;font-family:Calibri,Arial,sans-serif;font-size:11px;line-height:13px;mso-line-height-rule:exactly;'
    TH = 'background:#1e3a5f;color:#ffffff;padding:3px 8px;border:1px solid #2c4f7c;text-align:center;white-space:nowrap;font-weight:bold;'
    TD = 'padding:0 8px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#fff;'
    TC = 'padding:0 8px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#E3F2FD;font-weight:bold;'
    TT = 'padding:0 8px;border:1px solid #2c4f7c;text-align:center;white-space:nowrap;background:#1e3a5f;color:#ffffff;font-weight:bold;'

    hdr = '<tr>' + ''.join(f'<th style="{TH}"><p style="{P}color:#ffffff;font-weight:bold;">{c}</p></th>' for c in pivot.columns) + '</tr>'
    rows_html = ''
    for _, row in pivot.iterrows():
        cells = ''
        for col, val in zip(pivot.columns, row):
            st = TC if col == 'Count' else TD
            cells += f'<td style="{st}"><p style="{P}">{val}</p></td>'
        rows_html += f'<tr>{cells}</tr>'

    ncols = len(pivot.columns)
    tot_cells = ''.join(
        f'<td style="{TT}"><p style="{P}color:#fff;">{"Total" if idx == 0 else ""}</p></td>'
        for idx in range(ncols - 1)
    ) + f'<td style="{TT}"><p style="{P}color:#fff;">{total}</p></td>'

    return (
        f'<p style="font-weight:bold;font-family:Calibri,Arial,sans-serif;margin:16px 0 4px;font-size:13px;">{label}</p>'
        f'<table cellpadding="0" cellspacing="0" style="border-collapse:collapse;margin-bottom:14px;mso-table-lspace:0pt;mso-table-rspace:0pt;">'
        f'{hdr}{rows_html}<tr>{tot_cells}</tr></table>'
    )

# ── Helper: build detail HTML table ──────────────────────────
DETAIL_MAP = [
    ('emp id',          'Emp ID'),
    ('name',            'Name'),
    ('country',         'Country'),
    ('process',         'Process'),
    ('job family',      'Job Family'),
    ('lwd',             'LWD'),
    ('sup id',          'Sup ID'),
    ('supervisor name', 'Supervisor Name'),
    ('wd input',        'WD Input'),
    ('deactivated',     'Deactivated / Deleted (Y/N)'),
]

def build_detail_html(df, label):
    if df is None or df.empty: return ''
    col_map = {}
    for kw, disp in DETAIL_MAP:
        m = next((c for c in df.columns if kw.lower() in c.lower()), None)
        if m and m not in col_map:
            col_map[m] = disp
    if not col_map: return ''

    df_d = df[list(col_map.keys())].rename(columns=col_map).fillna('').astype(str)

    P  = 'margin:0;padding:0;font-family:Calibri,Arial,sans-serif;font-size:11px;line-height:13px;mso-line-height-rule:exactly;'
    TH = 'background:#1e3a5f;color:#ffffff;padding:3px 8px;border:1px solid #2c4f7c;text-align:center;white-space:nowrap;font-weight:bold;'
    TD = 'padding:0 8px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#fff;'
    TY = 'padding:0 8px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#C8E6C9;font-weight:bold;'

    hdr = '<tr>' + ''.join(f'<th style="{TH}"><p style="{P}color:#ffffff;font-weight:bold;">{c}</p></th>' for c in df_d.columns) + '</tr>'
    rows_html = ''
    for _, row in df_d.iterrows():
        cells = ''
        for val in row:
            st = TY if str(val).strip().upper() == 'Y' else TD
            cells += f'<td style="{st}"><p style="{P}">{val}</p></td>'
        rows_html += f'<tr>{cells}</tr>'

    return (
        f'<p style="font-weight:bold;font-family:Calibri,Arial,sans-serif;margin:16px 0 4px;font-size:13px;">{label} — Detail</p>'
        f'<table cellpadding="0" cellspacing="0" style="border-collapse:collapse;margin-bottom:14px;mso-table-lspace:0pt;mso-table-rspace:0pt;">'
        f'{hdr}{rows_html}</table>'
    )


# ── MAIN FLOW ─────────────────────────────────────────────────
print(f'[{datetime.now():%Y-%m-%d %H:%M:%S}] Starting SEA Pending Attrition auto-reply')

pythoncom.CoInitialize()
ol = win32com.client.Dispatch('Outlook.Application')
ns = ol.GetNamespace('MAPI')
ns.Logon()

print(f"[Search] Sender: '{SENDER_FILTER}' | Subject: '{SUBJECT_FILTER}'")
target_mail = None
tf = find_named_folder(ns.Folders, TARGET_FOLDER)
if tf:
    print(f"[Search] Folder '{tf.Name}' ({tf.Items.Count} items)")
    target_mail = search_folder(tf, SENDER_FILTER, SUBJECT_FILTER, DAYS_LOOKBACK)
if target_mail is None:
    print('[Search] Fallback: scanning all mailbox folders')
    for root in ns.Folders:
        try:
            target_mail = search_folder(root, SENDER_FILTER, SUBJECT_FILTER, DAYS_LOOKBACK)
            if target_mail: break
        except: continue

if target_mail is None:
    print('[Skip] No matching email found.')
else:
    print(f'[Found] {target_mail.Subject} | Received: {target_mail.ReceivedTime}')

    # Download attachment
    print(f'[Attachments] {target_mail.Attachments.Count} total')
    raw_path = None
    for att in target_mail.Attachments:
        if ATTACH_KEYWORD.lower() in att.FileName.lower():
            raw_path = SAVE_DIR / att.FileName
            att.SaveAsFile(str(raw_path))
            print(f'[Downloaded] {raw_path.name}')
            break

    if raw_path is None:
        print(f"[Skip] Attachment '{ATTACH_KEYWORD}' not found.")
    else:
        # Read + filter Excel
        xl     = pd.ExcelFile(str(raw_path))
        sheets = xl.sheet_names
        print(f'[Excel] Sheets: {sheets}')

        sheet_a = next((s for s in sheets if 'attrition' in s.lower()), None)
        sheet_t = next((s for s in sheets if 'transfer'  in s.lower()), None)

        df_a_raw = pd.read_excel(xl, sheet_name=sheet_a, dtype=str).fillna('') if sheet_a else pd.DataFrame()
        df_t_raw = pd.read_excel(xl, sheet_name=sheet_t, dtype=str).fillna('') if sheet_t else pd.DataFrame()
        xl.close()

        proc_col_a    = find_col(df_a_raw, 'process')
        country_col_a = find_col(df_a_raw, 'country')
        deact_col_a   = find_col(df_a_raw, 'deactivated', 'deleted')
        proc_col_t    = find_col(df_t_raw, 'old process', 'process')
        country_col_t = find_col(df_t_raw, 'country')
        deact_col_t   = find_col(df_t_raw, 'deactivated', 'deleted')

        def do_filter(df, proc_col, country_col):
            if df.empty or not proc_col: return pd.DataFrame()
            mask = df[proc_col].str.strip().str.lower() == PROCESS_FILTER.lower()
            if country_col:
                mask &= df[country_col].str.strip().str.lower() == COUNTRY_FILTER.lower()
            return df[mask].copy()

        df_a = do_filter(df_a_raw, proc_col_a, country_col_a)
        df_t = do_filter(df_t_raw, proc_col_t, country_col_t)

        # Strip time part from datetime strings (e.g. '2026-09-03 00:00:00' → '2026-09-03')
        _dt_pat = r' \d{2}:\d{2}:\d{2}(\.\d+)?$'
        df_a = df_a.replace(_dt_pat, '', regex=True)
        df_t = df_t.replace(_dt_pat, '', regex=True)

        total_cases = len(df_a) + len(df_t)
        print(f'[Filter] Attrition: {len(df_a)} | Transfer: {len(df_t)} | Total: {total_cases}')

        if total_cases == 0:
            print(f'[Skip] No {PROCESS_FILTER}+{COUNTRY_FILTER} cases found.')
        else:
            # Check already replied
            sent_folder = ns.GetDefaultFolder(5)
            replied, r_subj, r_time = already_replied(target_mail, sent_folder)
            if replied:
                print(f'[Skip] Already replied: {r_subj!r} at {r_time}')
            else:
                print('[Reply check] Not replied yet — proceeding')

                # Fill Y
                if deact_col_a and not df_a.empty: df_a[deact_col_a] = 'Y'
                if deact_col_t and not df_t.empty: df_t[deact_col_t] = 'Y'

                # Save file
                safe_subj = re.sub(r'[\\/:*?"<>|]', '-', str(target_mail.Subject)).strip()
                out_path  = SAVE_DIR / f'Expedia VN - {safe_subj}.xlsx'
                with pd.ExcelWriter(str(out_path), engine='openpyxl', mode='w') as writer:
                    if not df_a.empty: df_a.to_excel(writer, sheet_name=sheet_a or 'Pending Attrition', index=False)
                    if not df_t.empty: df_t.to_excel(writer, sheet_name=sheet_t or 'Pending Transfer',  index=False)
                print(f'[Saved] {out_path.name}')

                # Build HTML
                GROUP_A = ['country', 'location', 'process', 'job family', 'lob']
                GROUP_T = ['country', 'location', 'old process', 'new process', 'job family', 'lob']
                print('[Build] Building HTML tables...')
                pivot_section = build_pivot_html(df_a, f'Pending Attrition — {PROCESS_FILTER} {COUNTRY_FILTER}', GROUP_A)
                print('[Build] Attrition pivot done')
                pivot_section += build_pivot_html(df_t, f'Pending Transfer — {PROCESS_FILTER} {COUNTRY_FILTER}', GROUP_T)
                print('[Build] Transfer pivot done')
                pivot_section += build_detail_html(df_a, f'Pending Attrition — {PROCESS_FILTER} {COUNTRY_FILTER}')
                print('[Build] Attrition detail done')
                pivot_section += build_detail_html(df_t, f'Pending Transfer — {PROCESS_FILTER} {COUNTRY_FILTER}')
                print('[Build] All HTML done')

                FONT = 'font-family:Calibri,Arial,sans-serif;'
                email_body = (
                    f'<div style="{FONT}font-size:14px;color:#222;padding:16px 20px;">'
                    f'<p style="margin:0 0 10px;">Dear team,</p>'
                    f'<p style="margin:0 0 14px;">We have attached the file for accounts from '
                    f'<strong>{PROCESS_FILTER} {COUNTRY_FILTER}</strong>.<br>Please refer to the attached file.</p>'
                    + pivot_section +
                    f'<p style="font-size:13px;color:#444;margin:16px 0 0;border-top:1px solid #e0e0e0;padding-top:12px;line-height:1.8;">'
                    f'Thanks &amp; Regards,<br><strong>Chinh Nguyen</strong><br>'
                    f'Analyst, WFM Real Time Management<br>'
                    f'Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street,<br>'
                    f'Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam<br>'
                    f'Ph: +84 986 473 419&nbsp;|&nbsp;'
                    f'<a href="mailto:huuchinh.nguyen@concentrix.com" style="color:#1155CC;">huuchinh.nguyen@concentrix.com</a>'
                    f'</p></div>'
                )

                if DISPLAY_PREVIEW:
                    wrap = ('<!DOCTYPE html><html><head><meta charset="utf-8">'
                            '<style>body{margin:0;background:#e8e8e8;}.w{max-width:900px;margin:16px auto;background:#fff;border:1px solid #ccc;}</style>'
                            '</head><body><div class="w">' + email_body + '</div></body></html>')
                    esc = wrap.replace('&','&amp;').replace('"','&quot;').replace("'",'&#39;')
                    display(HTML(f'<iframe srcdoc="{esc}" style="width:100%;border:1px solid #ddd;min-height:400px;" '
                                 f'onload="this.style.height=(this.contentDocument.body.scrollHeight+30)+\'px\'"></iframe>'))

                # Reply-All
                print(f'[Reply] Creating ReplyAll...')
                reply = target_mail.ReplyAll()
                reply.To = EMAIL_TO
                reply.CC = EMAIL_CC

                raw     = reply.HTMLBody
                q_match = re.search(r'<div\s+id=["\']divRplyFwdMsg["\']', raw, re.IGNORECASE)
                quoted  = raw[q_match.start():] if q_match else ''
                if not quoted:
                    hr_m   = re.search(r'<hr\s[^>]*>', raw, re.IGNORECASE)
                    quoted = raw[hr_m.start():] if hr_m else ''

                reply.HTMLBody = email_body + (f'<div>{quoted}</div>' if quoted else '')
                reply.Attachments.Add(str(out_path.resolve()))
                print('[Reply] Sending...')
                reply.Send()
                time.sleep(2)
                print(f'[Done] Reply sent. Attrition: {len(df_a)} | Transfer: {len(df_t)}')


print(f'[{datetime.now():%Y-%m-%d %H:%M:%S}] Finished')

[2026-09-05 22:43:37] Starting SEA Pending Attrition auto-reply
[Search] Sender: 'SEA_WFM_ID_Deletion@concentrix.com' | Subject: 'SEA - Client ID Deletion Notification'
[Search] Folder 'GC3 + ExpAdmin' (183 items)
[Found] SEA - Client ID Deletion Notification (05-Sep-2026) | Received: 2026-09-05 11:32:56.229000+00:00
[Attachments] 1 total
[Downloaded] SEA - Pending Attrition- Transfer.xlsx
[Excel] Sheets: ['Pending Attrition', 'Pending Transfer']
[Filter] Attrition: 1 | Transfer: 0 | Total: 1
[Reply check] Not replied yet — proceeding
[Saved] Expedia VN - SEA - Client ID Deletion Notification (05-Sep-2026).xlsx
[Build] Building HTML tables...
[Build] Attrition pivot done
[Build] Transfer pivot done
[Build] Attrition detail done
[Build] All HTML done


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


[Reply] Creating ReplyAll...
[Reply] Sending...
[Done] Reply sent. Attrition: 1 | Transfer: 0
[2026-09-05 22:43:40] Finished
